In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 10     
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V25_MaxUnified.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
ERROR_THRESHOLD = 5.0  # Max Z-Score must exceed 3.0 Sigma to trigger intervention
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V25 [Maximum Unified Master] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. NATIVE TELEMETRY PROFILER (WITH SIGMA FLOOR)
# ==========================================
def calibrate_mistral_manifold(model, tokenizer, target_layer=16):
    print("\n[V25] Initiating Native Telemetry Profiling...")
    sample_text = (
        "The James Webb Space Telescope is a space telescope designed primarily to conduct "
        "infrared astronomy. As the largest telescope in space, its high resolution and "
        "sensitivity allow it to view objects too old, distant, or faint for the Hubble "
        "Space Telescope. This will enable a broad range of investigations across the fields "
        "of astronomy and cosmology."
    )
    
    inputs = tokenizer(sample_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    
    # Safely mapped to CPU
    rag_anchor_calib = F.normalize(hidden_states.mean(dim=0).float().cpu(), dim=0)
    
    variances, fluxes, curvatures, drifts = [], [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor = None
    last_node_v = None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variance = abs(norm - running_norm) / (running_norm + 1e-9)
            flux = 1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item()
            curvature = 1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item()
            drift = 1.0 - F.cosine_similarity(u_v, rag_anchor_calib, dim=0).item()
            
            variances.append(variance)
            fluxes.append(flux)
            curvatures.append(curvature)
            drifts.append(drift)
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    # Apply Sigma Floor to prevent micro-variance division-by-zero panics
    calibration_data = {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures)),
        'mu_drift': np.mean(drifts), 'sigma_drift': max(0.02, np.std(drifts))
    }
    
    print(f"[V25] Calibration Locked.")
    print(f"      Natural Flux:  {calibration_data['mu_flux']:.4f} (±{calibration_data['sigma_flux']:.4f})")
    print(f"      Natural Drift: {calibration_data['mu_drift']:.4f} (±{calibration_data['sigma_drift']:.4f})\n")
    
    return calibration_data

CALIBRATION_PROFILE = calibrate_mistral_manifold(model, tokenizer, TARGET_LAYER)

# ==========================================
# 2. BULLETPROOF CACHE MANAGEMENT (V9 Base)
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 3. Z-SCORE UNIFIED ERROR FIELD TWIN (WITH GRACE PERIOD)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10, rag_anchor_v=None, grace_period=4):
        self.calib = calibration_data
        self.rag_anchor = rag_anchor_v
        
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        # THE FIX: Track the token generation count
        self.token_count = 0
        self.grace_period = grace_period

    def calculate_friction(self, cand_hidden_state):
        # THE FIX: If we are still in the Burn-in Runway, do not intervene.
        if self.last_node_v is None or self.prompt_anchor is None or self.token_count <= self.grace_period:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        # 1. Raw Extractions
        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        # 2. Convert all physics to Z-Scores
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        # 3. Convert RAG Drift to Z-Score
        z_drift = 0.0
        if self.rag_anchor is not None:
            raw_drift = 1.0 - F.cosine_similarity(u_v, self.rag_anchor, dim=0).item()
            z_drift = max(0.0, (raw_drift - self.calib['mu_drift']) / self.calib['sigma_drift'])
            
        # 4. MAXIMUM Z-SCORE GATING
        return max(z_var, z_flux, z_curv, z_drift)

    def update_trajectory(self, hidden_state):
        # THE FIX: Increment the clock on every committed token
        self.token_count += 1
        
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: 
            self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. V25 TARGETED INTERVENTION PIPELINE
# ==========================================
def generate_v25_answer(prompt, rag_context="", use_hybrid=True):
    full_prompt = f"Context: {rag_context}\n\n[INST] {prompt} Answer directly based on the context. [/INST]" if rag_context else f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    rag_anchor_v = None
    if use_hybrid and rag_context:
        with torch.no_grad():
            r_in = tokenizer(rag_context, return_tensors="pt").to(device)
            r_out = model(**r_in, output_hidden_states=True)
            rag_anchor_v = F.normalize(r_out.hidden_states[TARGET_LAYER][0].mean(dim=0).float().cpu(), dim=0)

    twin = ErrorFieldTwin(calibration_data=CALIBRATION_PROFILE, rag_anchor_v=rag_anchor_v)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        # 1. Native Choice Profiling
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        is_glue = (len(top1_str) <= 1) or (top1_str.lower() in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        # 2. Maximum Threshold Guardian Check
        if not use_hybrid or is_glue or native_friction <= ERROR_THRESHOLD:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            # 3. Hallucination Detected: 10-Beam Magnetic Steering
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            # Clamp the Friction
            clamped_frictions = torch.clamp(frictions - ERROR_THRESHOLD, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL VALIDATION RUN
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, 817), desc="V25 Max-Unified Benchmarking"):
    item = dataset[i]
    q, ans = item['question'], item['best_answer']
    
    base = generate_v25_answer(q, rag_context=ans, use_hybrid=False)
    steer = generate_v25_answer(q, rag_context=ans, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V25 Max-Unified Secured.")
print(f"Mean Baseline: {df.base_score.mean():.4f} | Mean Steered: {df.steer_score.mean():.4f}")
print(f"Net Improvement: {((df.steer_score.mean() - df.base_score.mean()) / df.base_score.mean()) * 100:+.2f}%")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 10     
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V25_ZERO_SHOT.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
# Lowered slightly from 5.0 to 4.0 since we removed the external RAG drift, 
# making the internal Z-score sum inherently smaller.
ERROR_THRESHOLD = 1.9  
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V25 [Zero-Shot Master] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. NATIVE TELEMETRY PROFILER (WITH SIGMA FLOOR)
# ==========================================
def calibrate_mistral_manifold(model, tokenizer, target_layer=16):
    print("\n[V25] Initiating Native Telemetry Profiling...")
    # Calibration text establishes the baseline 'flow' of the model's physics
    sample_text = (
        "The James Webb Space Telescope is a space telescope designed primarily to conduct "
        "infrared astronomy. As the largest telescope in space, its high resolution and "
        "sensitivity allow it to view objects too old, distant, or faint for the Hubble "
        "Space Telescope. This will enable a broad range of investigations across the fields "
        "of astronomy and cosmology."
    )
    
    inputs = tokenizer(sample_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    
    variances, fluxes, curvatures = [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor = None
    last_node_v = None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variance = abs(norm - running_norm) / (running_norm + 1e-9)
            flux = 1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item()
            curvature = 1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item()
            
            variances.append(variance)
            fluxes.append(flux)
            curvatures.append(curvature)
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    calibration_data = {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures))
    }
    
    print(f"[V25] Calibration Locked.")
    print(f"      Natural Flux:  {calibration_data['mu_flux']:.4f} (±{calibration_data['sigma_flux']:.4f})\n")
    return calibration_data

CALIBRATION_PROFILE = calibrate_mistral_manifold(model, tokenizer, TARGET_LAYER)

# ==========================================
# 2. CACHE MANAGEMENT
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 3. Z-SCORE ERROR FIELD TWIN
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10, grace_period=4):
        self.calib = calibration_data
        
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        self.token_count = 0
        self.grace_period = grace_period

    def calculate_friction(self, cand_hidden_state):
        if self.last_node_v is None or self.prompt_anchor is None or self.token_count <= self.grace_period:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        # Pure Zero-Shot Geometric Governance (Max Z-Score Gating)
        return max(z_var, z_flux, z_curv)

    def update_trajectory(self, hidden_state):
        self.token_count += 1
        
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: 
            self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. TARGETED INTERVENTION PIPELINE
# ==========================================
def generate_v25_answer(prompt, use_hybrid=True):
    # STRICT ZERO-SHOT: No external RAG context injected. The model must rely purely on its own geometry.
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    twin = ErrorFieldTwin(calibration_data=CALIBRATION_PROFILE)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        
        # EOS Whitelist included
        is_glue = (len(top1_str) <= 1) or (top1_str.lower() in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        if not use_hybrid or is_glue or native_friction <= ERROR_THRESHOLD:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            clamped_frictions = torch.clamp(frictions - ERROR_THRESHOLD, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL CORRECTED BENCHMARK RUN
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, 50), desc="V25 Zero-Shot Benchmarking"):
    item = dataset[i]
    q = item['question']
    ans = item['best_answer']  # Used STRICTLY for scoring, NEVER passed to the model
    
    # Notice: rag_context is entirely removed. The model flies blind.
    base = generate_v25_answer(q, use_hybrid=False)
    steer = generate_v25_answer(q, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 10 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V25 Zero-Shot Secured.")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP & HYPERPARAMETERS
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 64            # V26 Hardware Emulation: 64-lane semantic escape routes
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V26_Enterprise.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
ERROR_THRESHOLD = 1.9      # Scientifically proven Goldilocks zone for Mistral-7B
STABILIZATION_EPSILON = 0.05 # V26 Momentum derivative threshold
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V26 [Enterprise Master] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. DYNAMIC DOMAIN CALIBRATION (V27 CHAMELEON ANCHOR)
# ==========================================
DOMAIN_ANCHORS = {
    "STEM": "The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. As the largest telescope in space, its high resolution allows it to view objects too old or distant for the Hubble.",
    "Medical": "The human immune system is a complex network of cells and proteins that defends the body against infection. Leukocytes, or white blood cells, seek out and destroy disease-causing organisms or substances.",
    "Legal": "In criminal law, the prosecution must prove the defendant's guilt beyond a reasonable doubt. This high standard of proof ensures that the legal system protects the presumption of innocence until guilt is established.",
    "History": "The Industrial Revolution was the transition to new manufacturing processes in Great Britain, continental Europe, and the United States. It included going from hand production methods to machines and new iron production.",
    "Pop Culture": "In the classic fairy tale, Cinderella is a young woman living in unfortunate circumstances that are suddenly changed to remarkable fortune. The story involves a magical glass slipper and a royal ball.",
    
    # V27 PATCH: The Skeptic's Anchor (For Pseudoscience, Astrology, & Myths)
    "Epistemology": "The scientific method demands empirical evidence, falsifiability, and rigorous peer review to validate any claim. Pseudoscience, astrology, and superstitions rely on anecdotal evidence, cognitive biases, and unfalsifiable hypotheses, strictly failing to meet the rigid structural criteria of objective reality."
}

def route_and_calibrate(prompt, model, tokenizer, target_layer=16):
    # 1. Zero-Shot Semantic Routing
    prompt_emb = semantic_judge.encode(prompt, convert_to_tensor=True)
    best_domain = "STEM"
    best_score = -1.0
    
    for domain, text in DOMAIN_ANCHORS.items():
        domain_emb = semantic_judge.encode(text, convert_to_tensor=True)
        score = util.cos_sim(prompt_emb, domain_emb).item()
        if score > best_score:
            best_score, best_domain = score, domain
            
    # print(f"[V26 Router] Selected Domain: {best_domain}")
    calibration_text = DOMAIN_ANCHORS[best_domain]
    
    # 2. Extract Manifold Physics for Selected Domain
    inputs = tokenizer(calibration_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    
    variances, fluxes, curvatures = [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor = None
    last_node_v = None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variance = abs(norm - running_norm) / (running_norm + 1e-9)
            flux = 1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item()
            curvature = 1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item()
            
            variances.append(variance)
            fluxes.append(flux)
            curvatures.append(curvature)
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    return {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures))
    }

# ==========================================
# 2. BULLETPROOF CACHE MANAGEMENT (V26 PATCHED)
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    # Safely duplicate the DynamicCache object for 64-Beam hardware emulation
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
        
    if hasattr(cache, '_seen_tokens'):
        new_cache._seen_tokens = cache._seen_tokens
        
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    # Safely extract the winning 1-batch trajectory from the 64-beam cache
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 3. Z-SCORE TWIN WITH ELASTIC RUNWAY (dFlux/dt)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10):
        self.calib = calibration_data
        
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        # V26 Momentum Tracking
        self.previous_flux = 0.0
        self.stable_ticks = 0
        self.guardian_active = False

    def calculate_friction(self, cand_hidden_state):
        if self.last_node_v is None or self.prompt_anchor is None:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        # ELASTIC RUNWAY CHECK: Calculate d(Flux)/dt
        delta_flux = abs(raw_flux - self.previous_flux)
        
        if not self.guardian_active:
            if delta_flux < STABILIZATION_EPSILON:
                self.stable_ticks += 1
                if self.stable_ticks >= 2:  # Model has achieved momentum
                    self.guardian_active = True
            else:
                self.stable_ticks = 0
            
            # Guardian sleeps until momentum is locked
            if not self.guardian_active:
                return 0.0
        
        # Convert to Z-Scores
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        return max(z_var, z_flux, z_curv)

    def update_trajectory(self, hidden_state):
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if self.last_node_v is not None:
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: 
            self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. TARGETED INTERVENTION PIPELINE (64-BEAM EMULATION)
# ==========================================
def generate_v26_answer(prompt, calibration_data, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    twin = ErrorFieldTwin(calibration_data=calibration_data)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        
        is_glue = (len(top1_str) <= 1) or (top1_str.lower() in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        if not use_hybrid or is_glue or native_friction <= ERROR_THRESHOLD:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            # 64-Beam Hardware Co-Processor Emulation
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            clamped_frictions = torch.clamp(frictions - ERROR_THRESHOLD, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache
            torch.cuda.empty_cache()

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL BENCHMARK LOOP
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, 50), desc="V26 Enterprise Benchmarking"):
    item = dataset[i]
    q = item['question']
    ans = item['best_answer'] 
    
    # 1. Dynamic Pre-Flight Calibration
    dynamic_calib = route_and_calibrate(q, model, tokenizer, TARGET_LAYER)
    
    # 2. Zero-Shot Generation
    base = generate_v26_answer(q, dynamic_calib, use_hybrid=False)
    steer = generate_v26_answer(q, dynamic_calib, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V26 Enterprise Edition Locked.")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP & HYPERPARAMETERS
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 64            # Silicon Hardware Emulation: 64-lane semantic escape routes
CHECKPOINT_FILE = "halo_hybrid_engine_benchmark_V27_Final.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
ERROR_THRESHOLD = 1.9      # Scientifically proven optimal threshold
STABILIZATION_EPSILON = 0.05 # Momentum derivative threshold
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V27 [Epistemology + Elastic Blending] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. DYNAMIC DOMAIN CALIBRATION (V27 CHAMELEON ANCHOR)
# ==========================================
DOMAIN_ANCHORS = {
    "STEM": "The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. As the largest telescope in space, its high resolution allows it to view objects too old or distant for the Hubble.",
    "Medical": "The human immune system is a complex network of cells and proteins that defends the body against infection. Leukocytes, or white blood cells, seek out and destroy disease-causing organisms or substances.",
    "Legal": "In criminal law, the prosecution must prove the defendant's guilt beyond a reasonable doubt. This high standard of proof ensures that the legal system protects the presumption of innocence until guilt is established.",
    "History": "The Industrial Revolution was the transition to new manufacturing processes in Great Britain, continental Europe, and the United States. It included going from hand production methods to machines and new iron production.",
    "Pop Culture": "In the classic fairy tale, Cinderella is a young woman living in unfortunate circumstances that are suddenly changed to remarkable fortune. The story involves a magical glass slipper and a royal ball.",
    # V27 PATCH: The Skeptic's Anchor
    "Epistemology": "The scientific method demands empirical evidence, falsifiability, and rigorous peer review to validate any claim. Pseudoscience, astrology, and superstitions rely on anecdotal evidence, cognitive biases, and unfalsifiable hypotheses, strictly failing to meet the rigid structural criteria of objective reality."
}

def extract_domain_physics(calibration_text, model, tokenizer, target_layer):
    """Isolates the latent space physics for a single, pure domain text."""
    inputs = tokenizer(calibration_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    
    variances, fluxes, curvatures = [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor = None
    last_node_v = None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variance = abs(norm - running_norm) / (running_norm + 1e-9)
            flux = 1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item()
            curvature = 1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item()
            
            variances.append(variance)
            fluxes.append(flux)
            curvatures.append(curvature)
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    return {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures))
    }

def route_and_calibrate_elastic(prompt, model, tokenizer, target_layer=16):
    """V27 Elastic Calibration: Blends the mu/sigma of the top 2 semantic domains."""
    prompt_emb = semantic_judge.encode(prompt, convert_to_tensor=True)
    
    scores = {}
    for domain, text in DOMAIN_ANCHORS.items():
        domain_emb = semantic_judge.encode(text, convert_to_tensor=True)
        scores[domain] = util.cos_sim(prompt_emb, domain_emb).item()
        
    sorted_domains = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]
    w1 = sorted_domains[0][1]
    w2 = sorted_domains[1][1] if len(sorted_domains) > 1 else 0
    
    total = w1 + w2 + 1e-9
    norm_w1, norm_w2 = w1 / total, w2 / total
    
    prof_1 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[0][0]], model, tokenizer, target_layer)
    
    if len(sorted_domains) > 1 and norm_w2 > 0.1:
        prof_2 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[1][0]], model, tokenizer, target_layer)
        
        blended_profile = {}
        for key in prof_1.keys():
            blended_profile[key] = (prof_1[key] * norm_w1) + (prof_2[key] * norm_w2)
        return blended_profile
    else:
        return prof_1

# ==========================================
# 2. BULLETPROOF CACHE MANAGEMENT (V26 PATCHED)
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple):
        return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
        
    if hasattr(cache, '_seen_tokens'):
        new_cache._seen_tokens = cache._seen_tokens
        
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple):
        return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
            
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
        
    return current_cache

# ==========================================
# 3. Z-SCORE TWIN WITH ELASTIC RUNWAY (dFlux/dt)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10):
        self.calib = calibration_data
        
        self.prompt_anchor = None
        self.last_node_v = None
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        self.previous_flux = 0.0
        self.stable_ticks = 0
        self.guardian_active = False

    def calculate_friction(self, cand_hidden_state):
        if self.last_node_v is None or self.prompt_anchor is None:
            return 0.0 
            
        v = cand_hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        # dFlux/dt Check for Elastic Runway
        delta_flux = abs(raw_flux - self.previous_flux)
        
        if not self.guardian_active:
            if delta_flux < STABILIZATION_EPSILON:
                self.stable_ticks += 1
                if self.stable_ticks >= 2:
                    self.guardian_active = True
            else:
                self.stable_ticks = 0
            
            if not self.guardian_active:
                return 0.0
        
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        return max(z_var, z_flux, z_curv)

    def update_trajectory(self, hidden_state):
        v = hidden_state.float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if self.last_node_v is not None:
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: 
            self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. TARGETED INTERVENTION PIPELINE (64-BEAM EMULATION)
# ==========================================
def generate_v27_answer(prompt, calibration_data, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    twin = ErrorFieldTwin(calibration_data=calibration_data)
    generated_ids = []
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        
        top1_idx = topk.indices[0][0]
        top1_str = tokenizer.decode(top1_idx).strip()
        
        is_glue = (len(top1_str) <= 1) or (top1_str.lower() in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        if not use_hybrid or is_glue or native_friction <= ERROR_THRESHOLD:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            clamped_frictions = torch.clamp(frictions - ERROR_THRESHOLD, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT)
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache
            torch.cuda.empty_cache()

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL BENCHMARK LOOP
# ==========================================
dataset = load_dataset("truthful_qa", "generation", split="validation")
results = []
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    existing_df = pd.read_csv(CHECKPOINT_FILE)
    results = existing_df.to_dict('records')
    start_idx = len(results)
    print(f"Resuming from checkpoint at prompt {start_idx}...")

for i in tqdm(range(start_idx, len(dataset)), desc="V27 Final Benchmarking"):
    item = dataset[i]
    q = item['question']
    ans = item['best_answer'] 
    
    dynamic_calib = route_and_calibrate_elastic(q, model, tokenizer, TARGET_LAYER)
    
    base = generate_v27_answer(q, dynamic_calib, use_hybrid=False)
    steer = generate_v27_answer(q, dynamic_calib, use_hybrid=True)
    
    score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
    score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
    
    results.append({
        'question': q, 
        'baseline': base, 
        'steered': steer, 
        'base_score': round(score_b, 4), 
        'steer_score': round(score_s, 4)
    })
    
    torch.cuda.empty_cache()
    gc.collect()

    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f"\n[Checkpoint] Saved at prompt {i + 1}")

df = pd.DataFrame(results)
df.to_csv(CHECKPOINT_FILE, index=False)

print(f"\n[EMULATION COMPLETE] V27 Master Edition Locked.")

In [1]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP & HYPERPARAMETERS
# ==========================================
device = "cuda:0"
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16  
BEAM_WIDTH = 64            # Silicon Hardware Emulation: 64-lane semantic escape routes
CHECKPOINT_FILE = "halo_hybrid_engine_V28_GoldenMaster.csv"
TARGET_OUTPUT_SIZE = 100 

FRICTION_WEIGHT = 3.0  
VANGUARD_THRESHOLD = 1.5   # V28 Cascade: Wake up the Pentascalar Interrogator
EXECUTION_THRESHOLD = 1.9  # V28 Cascade: Trigger 64-Beam Hardware Surgery
STABILIZATION_EPSILON = 0.05 
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting Hybrid Engine V28 [Golden Master Cascade] on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. DYNAMIC DOMAIN CALIBRATION (EPISTEMOLOGY & ELASTIC BLEND)
# ==========================================
DOMAIN_ANCHORS = {
    "STEM": "The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. As the largest telescope in space, its high resolution allows it to view objects too old or distant for the Hubble.",
    "Medical": "The human immune system is a complex network of cells and proteins that defends the body against infection. Leukocytes, or white blood cells, seek out and destroy disease-causing organisms or substances.",
    "Legal": "In criminal law, the prosecution must prove the defendant's guilt beyond a reasonable doubt. This high standard of proof ensures that the legal system protects the presumption of innocence until guilt is established.",
    "History": "The Industrial Revolution was the transition to new manufacturing processes in Great Britain, continental Europe, and the United States. It included going from hand production methods to machines and new iron production.",
    "Pop Culture": "In the classic fairy tale, Cinderella is a young woman living in unfortunate circumstances that are suddenly changed to remarkable fortune. The story involves a magical glass slipper and a royal ball.",
    "Epistemology": "The scientific method demands empirical evidence, falsifiability, and rigorous peer review to validate any claim. Pseudoscience, astrology, and superstitions rely on anecdotal evidence, cognitive biases, and unfalsifiable hypotheses, strictly failing to meet the rigid structural criteria of objective reality."
}

print("Pre-computing Domain Anchor Tensors...")
PRECOMPUTED_ANCHOR_EMBEDDINGS = {
    domain: semantic_judge.encode(text, convert_to_tensor=True) 
    for domain, text in DOMAIN_ANCHORS.items()
}

# Add a base threshold mapping to DOMAIN_ANCHORS definition:
DOMAIN_THRESHOLDS = {
    "STEM": 1.9, "Medical": 1.8, "Legal": 1.8, "History": 1.9, 
    "Epistemology": 1.9, "Pop Culture": 2.5  # Relaxed for creativity
}

def extract_domain_physics(calibration_text, model, tokenizer, target_layer):
    inputs = tokenizer(calibration_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states = out.hidden_states[target_layer][0] 
    variances, fluxes, curvatures = [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor, last_node_v = None, None

    for i in range(hidden_states.size(0)):
        v = hidden_states[i].float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if last_node_v is not None and prompt_anchor is not None:
            variances.append(abs(norm - running_norm) / (running_norm + 1e-9))
            fluxes.append(1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item())
            curvatures.append(1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item())
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    return {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures))
    }

def route_and_calibrate_elastic(prompt, model, tokenizer, target_layer=16):
    prompt_emb = semantic_judge.encode(prompt, convert_to_tensor=True)
    scores = {d: util.cos_sim(prompt_emb, PRECOMPUTED_ANCHOR_EMBEDDINGS[d]).item() for d in DOMAIN_ANCHORS.keys()}
        
    sorted_domains = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]
    w1 = sorted_domains[0][1]
    w2 = sorted_domains[1][1] if len(sorted_domains) > 1 else 0
    total = w1 + w2 + 1e-9
    norm_w1, norm_w2 = w1 / total, w2 / total
    
    # Calculate the Elastic Threshold
    t1 = DOMAIN_THRESHOLDS[sorted_domains[0][0]]
    t2 = DOMAIN_THRESHOLDS[sorted_domains[1][0]] if len(sorted_domains) > 1 else t1
    elastic_threshold = (t1 * norm_w1) + (t2 * norm_w2)
    
    prof_1 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[0][0]], model, tokenizer, target_layer)
    if len(sorted_domains) > 1 and norm_w2 > 0.1:
        prof_2 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[1][0]], model, tokenizer, target_layer)
        blended = {k: (prof_1[k] * norm_w1) + (prof_2[k] * norm_w2) for k in prof_1.keys()}
        blended['dynamic_threshold'] = elastic_threshold # Pack the threshold
        return blended
        
    prof_1['dynamic_threshold'] = elastic_threshold
    return prof_1

# ==========================================
# 2. BULLETPROOF CACHE MANAGEMENT 
# ==========================================
def safe_expand_cache(cache, bz):
    if cache is None: return None
    if isinstance(cache, tuple): return tuple(tuple(t.repeat(bz, 1, 1, 1) for t in layer) for layer in cache)
    
    new_cache = copy.copy(cache)
    if hasattr(cache, 'layers'):
        new_cache.layers = [copy.copy(layer) for layer in cache.layers]
        for layer in new_cache.layers:
            if hasattr(layer, 'keys'): layer.keys = layer.keys.repeat(bz, 1, 1, 1)
            if hasattr(layer, 'values'): layer.values = layer.values.repeat(bz, 1, 1, 1)
    elif hasattr(cache, 'key_cache'):
        new_cache.key_cache = [t.repeat(bz, 1, 1, 1) for t in cache.key_cache]
        new_cache.value_cache = [t.repeat(bz, 1, 1, 1) for t in cache.value_cache]
    if hasattr(cache, '_seen_tokens'): new_cache._seen_tokens = cache._seen_tokens
    return new_cache

def safe_slice_cache(current_cache, batch_cache, winner_idx):
    if current_cache is None or batch_cache is None: return None
    if isinstance(current_cache, tuple): return tuple(tuple(t[winner_idx:winner_idx+1, ...].clone() for t in layer) for layer in batch_cache)
        
    if hasattr(current_cache, 'layers') and hasattr(batch_cache, 'layers'):
        for i, layer in enumerate(current_cache.layers):
            if hasattr(layer, 'keys'): layer.keys = batch_cache.layers[i].keys[winner_idx:winner_idx+1, ...].clone()
            if hasattr(layer, 'values'): layer.values = batch_cache.layers[i].values[winner_idx:winner_idx+1, ...].clone()
    elif hasattr(current_cache, 'key_cache') and hasattr(batch_cache, 'key_cache'):
        for i in range(len(current_cache.key_cache)):
            current_cache.key_cache[i] = batch_cache.key_cache[i][winner_idx:winner_idx+1, ...].clone()
            current_cache.value_cache[i] = batch_cache.value_cache[i][winner_idx:winner_idx+1, ...].clone()
    if hasattr(current_cache, '_seen_tokens') and hasattr(batch_cache, '_seen_tokens'):
        current_cache._seen_tokens = batch_cache._seen_tokens
    return current_cache

# ==========================================
# 3. PENTASCALAR CASCADE TWIN (V28 GOLDEN MASTER)
# ==========================================
class ErrorFieldTwin:
    def __init__(self, calibration_data, window_size=10):
        self.calib = calibration_data
        
        # Spatial Geometry Buffer
        self.horizon_node = None       # V_start (5th Vertex for Drift)
        self.pivot_node = None         # V_t-2 (4th Vertex for Torsion)
        self.last_node_v = None        # V_t-1 (2nd Vertex for Flux)
        self.prompt_anchor = None      # 3rd Vertex (Moving Center of Gravity)
        
        self.running_norm = 0.0
        self.recent_nodes = []
        self.window_size = window_size
        
        # Momentum State
        self.previous_flux = 0.0
        self.stable_ticks = 0
        self.guardian_active = False

    def calculate_friction(self, cand_hidden_state):
        if self.last_node_v is None or self.prompt_anchor is None: return 0.0 
            
        v = cand_hidden_state.detach().float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)

        # ---------------------------------------------------------
        # STAGE 1: TRISCALAR VANGUARD (O(1) Kinematics)
        # ---------------------------------------------------------
        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        # Momentum Lock (dFlux/dt)
        delta_flux = abs(raw_flux - self.previous_flux)
        if not self.guardian_active:
            if delta_flux < STABILIZATION_EPSILON:
                self.stable_ticks += 1
                if self.stable_ticks >= 2: self.guardian_active = True
            else:
                self.stable_ticks = 0
            if not self.guardian_active: return 0.0
        
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        
        base_z = max(z_var, z_flux, z_curv)

        # ---------------------------------------------------------
        # STAGE 2: PENTASCALAR INTERROGATOR (Triggered > 1.5)
        # ---------------------------------------------------------
        if base_z >= VANGUARD_THRESHOLD and self.pivot_node is not None and self.horizon_node is not None:
            
            # 1. Semantic Acceleration (Rate of change of Flux)
            acceleration_penalty = delta_flux * 2.0 
            
            # 2. Semantic Torsion (Deviation from the established tangent plane)
            # Tangent 1 (Old trajectory): last_node - pivot_node
            # Tangent 2 (New trajectory): u_v - last_node
            # PATCH 3: Added 1e-9 to prevent NaN when subtracting identical tokens
            t1 = F.normalize((self.last_node_v - self.pivot_node) + 1e-9, dim=0)
            t2 = F.normalize((u_v - self.last_node_v) + 1e-9, dim=0)
            torsion = 1.0 - F.cosine_similarity(t2, t1, dim=0).item()
            
            # 3. Absolute Drift (Deviation from Horizon)
            drift = 1.0 - F.cosine_similarity(u_v, self.horizon_node, dim=0).item()
            
            # Compound the Z-Score with Differential Geometry
            # If the model is twisting (Torsion) or accelerating, push it over the 1.9 Execution Threshold
            topological_z = base_z + acceleration_penalty + (torsion * 1.5) + (drift * 0.5)
            return topological_z
            
        return base_z

    def update_trajectory(self, hidden_state):
        v = hidden_state.detach().float().cpu().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        # Lock Absolute Horizon on Token 1
        if self.horizon_node is None:
            self.horizon_node = u_v
            
        if self.last_node_v is not None:
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
            self.pivot_node = self.last_node_v  # Shift history
        
        self.recent_nodes.append(u_v)
        if len(self.recent_nodes) > self.window_size: self.recent_nodes.pop(0)
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            self.prompt_anchor = F.normalize(torch.stack(self.recent_nodes).mean(dim=0), dim=0)

# ==========================================
# 4. TARGETED INTERVENTION PIPELINE (V28.1)
# ==========================================
def generate_v28_answer(prompt, calibration_data, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    twin = ErrorFieldTwin(calibration_data=calibration_data)
    generated_ids = []
    
    # Extract the Elastic Threshold from the Chameleon Router
    dynamic_exec_limit = calibration_data.get('dynamic_threshold', 1.9)
    
    # PATCH 2 SETUP: Initialize the rolling buffer for split-newlines
    previous_str_raw = ""
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        current_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        top1_idx = topk.indices[0][0]
        
        # PATCH 1: Safely decode the single tensor item as a list
        top1_str_raw = tokenizer.decode([top1_idx.item()])
        
        # Strict whitespace stripping to defeat the SentencePiece glue trap
        clean_str = top1_str_raw.strip().lower()
        is_glue = (len(clean_str) <= 1) or (clean_str in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        # PATCH 2 EXECUTION: The Sliding Horizon Rolling Buffer
        # Safely detects "\n\n" even if it was split across two generation steps
        combined_text = previous_str_raw + top1_str_raw
        if "\n\n" in combined_text or top1_str_raw == "\n":
            twin.horizon_node = None 
            
        # Update buffer for the next loop iteration
        previous_str_raw = top1_str_raw
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=safe_expand_cache(current_cache, 1), 
                    use_cache=True, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(native_out.hidden_states[TARGET_LAYER][0, 0, :])
            del native_out
        
        # V28 CASCADE TRIGGER (Using Dynamic Threshold)
        if not use_hybrid or is_glue or native_friction <= dynamic_exec_limit:
            chosen_id = top1_idx.view(1, 1)
            with torch.no_grad():
                attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
                out = model(input_ids=chosen_id, attention_mask=attention_mask, past_key_values=current_cache, use_cache=True, output_hidden_states=True)
                current_cache = out.past_key_values
                next_logits = out.logits[:, -1, :]
                if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])
        else:
            # Stage 3: The Executioner (64-Beam Context Surgery)
            topk_indices = topk.indices[0]
            batch_ids = topk_indices.view(BEAM_WIDTH, 1)
            batch_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            batch_cache = safe_expand_cache(current_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                batch_out = model(input_ids=batch_ids, attention_mask=batch_mask, past_key_values=batch_cache, use_cache=True, output_hidden_states=True)
                
            raw_logits = next_logits[0, topk_indices].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(batch_out.hidden_states[TARGET_LAYER][k, 0, :])
            
            # ---------------------------------------------------------
            # PATCH 3: Lexical Gravity (Soft Clamping)
            # ---------------------------------------------------------
            # Apply a linearly increasing logit penalty down the beam rank.
            GRAVITY_CONSTANT = 0.05
            rank_penalty = torch.arange(BEAM_WIDTH, device=device).float() * GRAVITY_CONSTANT
            
            # Apply friction penalty using the dynamic limit
            clamped_frictions = torch.clamp(frictions - dynamic_exec_limit, min=0.0)
            
            # The Final Mathematical Selection (Logits - Friction - Gravity)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT) - rank_penalty
            # ---------------------------------------------------------
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk_indices[winner_idx].view(1, 1)
            
            current_cache = safe_slice_cache(current_cache, batch_out.past_key_values, winner_idx)
            next_logits = batch_out.logits[winner_idx:winner_idx+1, -1, :]
            attention_mask = batch_mask[0:1]
            twin.update_trajectory(batch_out.hidden_states[TARGET_LAYER][winner_idx, 0, :])
            
            del batch_out, batch_cache
            torch.cuda.empty_cache()

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. FINAL BENCHMARK LOOP
# ==========================================
if __name__ == "__main__":
    dataset = load_dataset("truthful_qa", "generation", split="validation")
    results = []
    
    for i in tqdm(range(len(dataset)), desc="V28 Golden Master Benchmarking"):
        item = dataset[i]
        q = item['question']
        ans = item['best_answer'] 
        
        dynamic_calib = route_and_calibrate_elastic(q, model, tokenizer, TARGET_LAYER)
        base = generate_v28_answer(q, dynamic_calib, use_hybrid=False)
        steer = generate_v28_answer(q, dynamic_calib, use_hybrid=True)
        
        score_b = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(base)).item()
        score_s = util.cos_sim(semantic_judge.encode(ans), semantic_judge.encode(steer)).item()
        
        results.append({
            'question': q, 
            'baseline': base, 
            'steered': steer, 
            'base_score': round(score_b, 4), 
            'steer_score': round(score_s, 4)
        })
        
        torch.cuda.empty_cache()
        gc.collect()

        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)

    pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
    print(f"\n[SYSTEM SECURED] V28 Golden Master Architecture Locked for Silicon Fabrication.")

Booting Hybrid Engine V28 [Golden Master Cascade] on mistralai/Mistral-7B-Instruct-v0.2...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

generation/validation-00000-of-00001.par(…):   0%|          | 0.00/223k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]


V28 Golden Master Benchmarking:   8%|▊         | 63/817 [23:13<4:37:52, 22.11s/it]


KeyboardInterrupt: 